# TaxaLens v0.6 — BIOSCAN Diptera 30k (selective)

Этот notebook готовит **ровно 30 000 Diptera** из официального BIOSCAN-5M до начала foundation training. Он скачивает полные metadata, но **не скачивает полные image ZIP**: из архивов читаются только выбранные JPEG через HTTP Range.

Прогресс и изображения сохраняются на Google Drive. Если Colab остановится, снова запусти те же ячейки сверху вниз — целые JPEG будут пропущены.

In [ ]:
# 1. Подключаем постоянное хранилище
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Берём актуальный TaxaLens из GitHub и ставим зависимости
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/SaniyaSani/TaxaLens.git'
PROJECT = Path('/content/TaxaLens')
if (PROJECT / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT)], check=True)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-foundation.txt'], check=True)
print('TaxaLens ready:', PROJECT)

In [ ]:
# 3. Все большие файлы остаются на Drive
STORE = Path('/content/drive/MyDrive/TaxaLens/Foundation_v06/raw/bioscan')
STORE.mkdir(parents=True, exist_ok=True)
print('BIOSCAN store:', STORE)
print('Free Drive space (GB):', round(__import__('shutil').disk_usage(STORE).free / 1024**3, 1))

## Этап A — metadata и неизменяемый список 30k

Эта ячейка скачивает только официальный metadata archive, проверяет checksum и делает два потоковых прохода по таблице. JPEG пока не скачиваются. Обычно это занимает заметно меньше времени и места, чем images.

In [ ]:
# 4. Выбрать ровно 30 000 Diptera по metadata
command = [
    sys.executable, 'scripts/run_bioscan_30k.py',
    '--root', str(STORE),
    '--max-records', '30000',
    '--max-per-taxon', '500',
    '--selection-only',
]
subprocess.run(command, cwd=PROJECT, check=True)
print('SELECTION READY')

In [ ]:
# 5. Проверить, что выборка действительно равна 30k и увидеть coverage
import json, pandas as pd
report_path = STORE / 'diptera_30k_selection_report.json'
report = json.loads(report_path.read_text())
display(pd.DataFrame([report['selected_by_split']]).T.rename(columns={0: 'images'}))
print('selected:', report['selected'])
print('taxonomic coverage:', report['taxonomic_coverage'])
assert report['selected'] == 30000, report
print('✓ BIOSCAN Diptera selection is exactly 30,000')

## Этап B — только выбранные JPEG

Эта ячейка может работать долго: она индексирует большие удалённые ZIP, но сам ZIP целиком не сохраняет. Строка `full_archives_downloaded: false` в отчёте подтверждает selective mode. Остановку можно пережить: перезапусти ячейку, и валидные существующие JPEG будут пропущены.

In [ ]:
# 6. Скачать только 30 000 выбранных JPEG и собрать нормализованный Parquet
command = [
    sys.executable, 'scripts/run_bioscan_30k.py',
    '--root', str(STORE),
    '--max-records', '30000',
    '--max-per-taxon', '500',
]
subprocess.run(command, cwd=PROJECT, check=True)
print('BIOSCAN 30k DOWNLOAD READY')

In [ ]:
# 7. Финальная строгая проверка перед foundation merge/training
manifest_path = STORE / 'bioscan_diptera_30k_manifest.parquet'
download_report = json.loads((STORE / 'diptera_30k_download_report.json').read_text())
manifest = pd.read_parquet(manifest_path)
assert download_report['complete'] == 30000, download_report
assert download_report['full_archives_downloaded'] is False
assert len(manifest) == 30000
assert set(manifest['order'].dropna().str.casefold()) == {'diptera'}
assert manifest['local_path'].map(lambda p: Path(p).is_file()).all()
foundation_manifest = STORE.parents[1] / 'manifests' / 'bioscan_raw.parquet'
foundation_manifest.parent.mkdir(parents=True, exist_ok=True)
__import__('shutil').copy2(manifest_path, foundation_manifest)
display(manifest.groupby(['source', 'source_split']).size().rename('images').to_frame())
print('✓ READY FOR MASTER CORPUS:', foundation_manifest)

## Если сервер вернул range-ошибку

В ячейке 6 добавь в `command` строку `'--no-suffix-range'` и запусти снова. Уже загруженные JPEG сохранятся.

После зелёной проверки выше BIOSCAN-часть готова. Следующий этап — собрать такие же manifests iNaturalist, GBIF и DiSSCo, объединить их без дубликатов и только затем запускать DINOv2 embedding shards.